# 🤖 Curso de Big Data e IA — Módulo 2
## Bloques 5-6: Fundamentos de IA y Machine Learning aplicado

**Duración estimada:** 6 horas · **Caso práctico:** predecir si un tiro acaba en gol (un modelo de *expected goals*)

**Datos:** [StatsBomb Open Data](https://github.com/statsbomb/open-data). Si publicas o compartes análisis basados en estos datos, indica StatsBomb como fuente y usa su logo (disponible en su *media pack*).

**Objetivos**
- Diferenciar IA, Machine Learning y Deep Learning, y aprendizaje supervisado y no supervisado.
- Construir variables (*feature engineering*) a partir de datos de posición.
- Entrenar y evaluar un modelo de clasificación con datos desbalanceados.
- Comparar nuestro modelo con el xG profesional de StatsBomb.
- Reflexionar sobre ética y sesgos.

**Antes de empezar:** sube a Colab `sporting_eventos.csv` (generado en el Notebook 1).

## 1. Fundamentos de IA

- **IA**: máquinas que realizan tareas que requieren "inteligencia".
- **Machine Learning**: la máquina **aprende patrones a partir de datos** en lugar de seguir reglas escritas a mano.
- **Deep Learning**: ML con redes neuronales de muchas capas; brilla con imágenes, texto o audio.

**Tipos de aprendizaje**
- **Supervisado:** hay una etiqueta conocida (¿acabó en gol?). Es lo que haremos.
- **No supervisado:** no hay etiqueta; el modelo busca grupos (por ejemplo, zonas de remate). También veremos un ejemplo.

**Problema de hoy:** dado un tiro (dónde, con qué parte del cuerpo, si había presión...), ¿cuál es la probabilidad de gol?

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Circle
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans
from sklearn.metrics import (roc_auc_score, confusion_matrix, ConfusionMatrixDisplay,
                             classification_report, RocCurveDisplay)

try:
    eventos = pd.read_csv("sporting_eventos.csv")
except FileNotFoundError:
    raise SystemExit("Sube sporting_eventos.csv (generado en el Notebook 1) al panel de archivos de Colab.")

tiros = eventos[eventos["type"] == "Shot"].copy()
print(f"Tiros: {len(tiros)}  |  Goles: {(tiros['shot_outcome'] == 'Goal').sum()}")

## 2. Preparación: la variable objetivo y el desbalanceo

Nuestra etiqueta es `gol` (1 si el tiro es gol, 0 si no). Fíjate en cuántos hay de cada.

In [ ]:
tiros["gol"] = (tiros["shot_outcome"] == "Goal").astype(int)
print(tiros["gol"].value_counts())
print(f"\nPorcentaje de goles: {tiros['gol'].mean() * 100:.1f}%")

Solo alrededor de **uno de cada diez tiros es gol**. Es un problema **desbalanceado**: un modelo que dijera siempre "no será gol"
acertaría ~90% sin aprender nada. Por eso **el *accuracy* engaña** y usaremos otras métricas (AUC, recall, matriz de confusión).

## 3. Feature engineering: de coordenadas a geometría

El modelo no entiende "x=108, y=35". Pero sabemos que importan dos cosas: la **distancia a la portería** y el **ángulo de tiro**
(cuánto se ve de la portería). La portería está en `x=120`, entre `y=36` y `y=44`.

In [ ]:
dx = 120 - tiros["x"]
dy = tiros["y"] - 40

tiros["distancia"] = np.sqrt(dx**2 + dy**2)
# Ángulo bajo el que se ve la portería (8 m de ancho)
tiros["angulo"] = np.arctan2(8 * dx, dx**2 + dy**2 - 16)

tiros.groupby("gol")[["distancia", "angulo"]].mean().round(2)

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 4))
tiros.boxplot("distancia", by="gol", ax=axs[0]); axs[0].set_title("Distancia a portería (m)")
tiros.boxplot("angulo", by="gol", ax=axs[1]); axs[1].set_title("Ángulo de tiro (rad)")
plt.suptitle("")
plt.show()

Los goles se marcan desde **más cerca** y con **más ángulo**. El modelo debería descubrirlo solo.

Añadimos variables categóricas (parte del cuerpo, tipo de tiro, jugada, presión) convertidas a números con *one-hot encoding*.

In [ ]:
base = tiros[["distancia", "angulo", "shot_body_part", "shot_type", "play_pattern"]].copy()
base["bajo_presion"] = tiros["under_pressure"].astype(int)
base["de_primeras"] = tiros["shot_first_time"].fillna(False).astype(int)

X = pd.get_dummies(base, columns=["shot_body_part", "shot_type", "play_pattern"], drop_first=True)
y = tiros["gol"]

print(f"{X.shape[1]} variables predictoras")
X.head(3)

## 4. División entrenamiento / test

Nunca evaluamos con los mismos datos que usamos para entrenar. Con `stratify=y` mantenemos la proporción de goles en ambos conjuntos.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
print(f"Entrenamiento: {len(X_train)} tiros ({y_train.sum()} goles)")
print(f"Test:          {len(X_test)} tiros ({y_test.sum()} goles)")

## 5. Primer modelo: regresión logística

Un modelo simple e interpretable. Usamos `class_weight="balanced"` para que no ignore los goles, que son pocos.

In [ ]:
lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, class_weight="balanced"))
lr.fit(X_train, y_train)

prob_lr = lr.predict_proba(X_test)[:, 1]
print(f"AUC (regresión logística): {roc_auc_score(y_test, prob_lr):.3f}")

**AUC** mide si el modelo ordena bien: dado un tiro que fue gol y otro que no, ¿asigna más probabilidad al gol?
0,5 es azar y 1 es perfecto.

## 6. Segundo modelo: Random Forest

In [ ]:
rf = RandomForestClassifier(n_estimators=300, min_samples_leaf=5, class_weight="balanced", random_state=42)
rf.fit(X_train, y_train)

prob_rf = rf.predict_proba(X_test)[:, 1]
print(f"AUC (Random Forest): {roc_auc_score(y_test, prob_rf):.3f}")

## 7. Evaluación: matriz de confusión y métricas

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(11, 4))
for ax, (nombre, prob) in zip(axs, [("Regresión logística", prob_lr), ("Random Forest", prob_rf)]):
    cm = confusion_matrix(y_test, (prob > 0.5).astype(int))
    ConfusionMatrixDisplay(cm, display_labels=["No gol", "Gol"]).plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(nombre)
plt.tight_layout()
plt.show()

print(classification_report(y_test, (prob_rf > 0.5).astype(int), target_names=["No gol", "Gol"]))

**Cómo leerlo:**
- **Recall de "Gol"**: de los goles reales, ¿cuántos detectó el modelo?
- **Precisión de "Gol"**: de los tiros que dijo que serían gol, ¿cuántos lo fueron? Suele ser baja: es normal, porque la mayoría de los tiros no acaban en gol.

Con el umbral 0,5 el modelo da muchos falsos positivos. El umbral es una **decisión de negocio**: ¿preferimos no dejar escapar ninguna buena ocasión o evitar falsas alarmas?

## 8. ¿Nuestro modelo es bueno? Comparación con el xG de StatsBomb

StatsBomb ya calcula su propio xG con más información (por ejemplo, la posición de los defensores).

In [ ]:
auc_sb = roc_auc_score(y_test, tiros.loc[y_test.index, "shot_xg"])
print(f"AUC regresión logística: {roc_auc_score(y_test, prob_lr):.3f}")
print(f"AUC Random Forest:       {roc_auc_score(y_test, prob_rf):.3f}")
print(f"AUC xG de StatsBomb:     {auc_sb:.3f}")

fig, ax = plt.subplots(figsize=(6, 5))
RocCurveDisplay.from_predictions(y_test, prob_lr, name="Reg. logística", ax=ax)
RocCurveDisplay.from_predictions(y_test, prob_rf, name="Random Forest", ax=ax)
RocCurveDisplay.from_predictions(y_test, tiros.loc[y_test.index, "shot_xg"], name="xG StatsBomb", ax=ax)
plt.show()

Nuestro modelo, con muy pocas variables y menos de mil tiros, se acerca al profesional pero no lo iguala.
La diferencia se explica sobre todo por la **información**: el modelo de StatsBomb usa datos que nosotros no tenemos.

## 9. ¿Y la fiabilidad? Validación cruzada

Con pocos goles, el resultado depende mucho de qué tiros caen en el test. La validación cruzada repite el experimento 5 veces.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
auc_cv = cross_val_score(rf, X, y, cv=cv, scoring="roc_auc")

print("AUC en cada vuelta:", auc_cv.round(3))
print(f"Media: {auc_cv.mean():.3f}  |  Desviación: {auc_cv.std():.3f}")
print(f"\nxG de StatsBomb sobre todos los tiros (no se entrena): {roc_auc_score(y, tiros['shot_xg']):.3f}")

Fíjate en la **desviación**: el AUC cambia bastante de una vuelta a otra. Con solo un centenar de goles, el resultado depende mucho de qué tiros caen en el test (por eso el AUC de la sección 8 puede variar al cambiar la semilla). Conclusión: con pocos datos, hay que desconfiar de un único número.

## 10. Variables más influyentes

In [ ]:
imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False).head(8)
imp.plot(kind="barh", color="#3FB8AF")
plt.gca().invert_yaxis()
plt.title("Variables más importantes del Random Forest")
plt.show()

## 11. Aprendizaje no supervisado: zonas de remate

Ahora **sin etiqueta**: ¿qué zonas del campo agrupa `KMeans` solo con la posición de los tiros? Después miramos qué porcentaje de goles hay en cada zona.

In [ ]:
km = KMeans(n_clusters=4, n_init=10, random_state=42)
tiros["zona"] = km.fit_predict(tiros[["x", "y"]])

resumen_zonas = tiros.groupby("zona").agg(tiros=("gol", "size"), goles=("gol", "sum"), xg_medio=("shot_xg", "mean")).round(3)
resumen_zonas["% goles"] = (resumen_zonas["goles"] / resumen_zonas["tiros"] * 100).round(1)
resumen_zonas

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6.5))
ax.set_xlim(55, 122); ax.set_ylim(82, -2)
ax.add_patch(Rectangle((60, 0), 60, 80, fill=False, ec="#555"))
ax.add_patch(Rectangle((102, 18), 18, 44, fill=False, ec="#555"))
ax.add_patch(Rectangle((114, 30), 6, 20, fill=False, ec="#555"))
sc = ax.scatter(tiros["x"], tiros["y"], c=tiros["zona"], cmap="Set2", alpha=0.8)
ax.set_aspect("equal"); ax.axis("off")
ax.set_title("Zonas de remate encontradas por KMeans")
plt.show()

## 12. Redes neuronales (nota conceptual)

Para datos de tabla y pocos ejemplos, un Random Forest suele rendir igual o mejor que una red neuronal y es más fácil de explicar.
Probamos una red pequeña solo para comparar.

In [ ]:
from sklearn.neural_network import MLPClassifier

nn = make_pipeline(StandardScaler(), MLPClassifier(hidden_layer_sizes=(16, 8), max_iter=1000, random_state=42))
nn.fit(X_train, y_train)
print(f"AUC (red neuronal simple): {roc_auc_score(y_test, nn.predict_proba(X_test)[:, 1]):.3f}")

## 13. Ética y sesgos

- **Datos pequeños:** menos de mil tiros y unos cien goles de una sola temporada. Las conclusiones sobre un jugador concreto serían poco fiables.
- **Sesgo de la muestra:** solo hay tiros de los partidos del Sporting; los rivales aparecen solo contra él.
- **Uso de datos de personas:** son datos de rendimiento de futbolistas profesionales. Aunque sean públicos, usarlos para decisiones sobre contratos o salarios exige cuidado, transparencia y respetar la normativa de protección de datos.
- **Falsos positivos y negativos:** si un modelo se usara para fichajes, ¿qué cuesta más, descartar a un buen jugador o fichar a uno que no lo es?
- **Atribución:** indica siempre StatsBomb como fuente si compartes resultados.

## 📝 Ejercicio propuesto

1. Cambia `random_state` en `train_test_split` y observa cuánto varía el AUC. ¿Qué te dice sobre la fiabilidad?
2. Entrena el modelo **sin** `distancia` y `angulo`. ¿Cuánto empeora?
3. Prueba otro umbral (por ejemplo 0,3 y 0,7). ¿Cómo cambian recall y precisión de "Gol"?
4. ¿Qué otras variables añadirías si las tuvieras? (pista: ¿dónde estaban los defensores?)